In [1]:
import torch
from dataLoader import *
from joblib import parallel_backend
from skorch import NeuralNetRegressor

def resize_tensor(x,size = (7,7)):
    x = x.unsqueeze(0)
    x = torch.nn.functional.interpolate(x, size=size, mode='bilinear', align_corners=False)
    return x.squeeze(0)
    
def load_images(dirName):
    files_train = sorted(dirName.glob("*.npz"))
    train_data = []
    for file in enumerate(files_train):
        file_name = file[1]
        with np.load(file_name) as npz:
            arr = np.ma.MaskedArray(**npz)
            data = arr.data
            data = np.append(data, [arr.mask[0]], axis=0)
            x = torch.tensor(data, dtype=torch.float32)
            x = resize_tensor(x)
            train_data.append(x)
        
    max_width = max(data.shape[1] for data in train_data)
    max_height = max(data.shape[2] for data in train_data)
    max_size = max_width if max_width > max_height else max_height

    return [resize_tensor(data) for data in train_data]
    
msi = load_images(MSI_SATELLITE_TRAIN_DIR)
hsi = load_images(HSI_SATELLITE_DIR)
airborne = load_images(HSI_AIRBORNE_DIR)
y_train = gt_train_df[column_names].values
y_train = torch.tensor(y_train, dtype=torch.float32)
device = "cuda" if torch.cuda.is_available() else "cpu"

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


In [2]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(msi, y_train, train_size=0.8)

In [3]:
from torch.utils.data import Dataset

class MultiSourceDataset(Dataset):
    def __init__(self, msi, hsi, airborne, targets):
        self.msi = msi
        self.hsi = hsi
        self.airborne = airborne
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return (
            self.msi[idx],
            self.hsi[idx],
            self.airborne[idx],
            self.targets[idx]
        )

In [4]:
param_grid_2D = {
    "module__neurons":[32,64,128],
    "lr":[1e-3, 1e-4, 1e-5],
}

param_grid_3D = {
    "regressor__neurons":[32,64,128],
    "regressor__kernel_size":[3,4,5],
    "lr":[1e-3, 1e-4, 1e-5],
}

# 2D Kernel

In [5]:
from skorch import NeuralNetRegressor
from sklearn.model_selection import GridSearchCV
from neural_networks.Convolutional import CNN2D
from joblib import parallel_backend
import time

models = []

for i in range(y_train.shape[1]):
    print(f"Training model for column {i}")

    y_col = y_train[:, i]

    net = NeuralNetRegressor(
        module=CNN2D(neurons=64),
        max_epochs=50,
        lr=1e-3,
        batch_size=128,
        optimizer=torch.optim.Adam,
        device=device,
        verbose=0
    )

    gs = GridSearchCV(
        net,
        param_grid_2D,
        cv=3,
        scoring="neg_mean_squared_error",
        verbose=2,
        n_jobs=-1
    )

    start = time.perf_counter()
    with parallel_backend("threading"):
        gs.fit(X_train, y_col)
    end = time.perf.counter()
    print(f"time: {end-start:4f}")

    print(f"Best params for column {i}: {gs.best_params_}")
    models.append(gs.best_estimator_)

Training model for column 0
Fitting 3 folds for each of 9 candidates, totalling 27 fits
[CV] END ......................lr=0.0001, module__neurons=64; total time=   0.8s
[CV] END .......................lr=0.001, module__neurons=64; total time=   0.8s
[CV] END .......................lr=0.001, module__neurons=64; total time=   0.8s
[CV] END .......................lr=0.001, module__neurons=32; total time=   0.8s
[CV] END .......................lr=0.001, module__neurons=32; total time=   0.8s
[CV] END ......................lr=0.0001, module__neurons=32; total time=   0.8s
[CV] END ......................lr=0.0001, module__neurons=32; total time=   0.8s
[CV] END .......................lr=0.001, module__neurons=64; total time=   0.8s
[CV] END .....................lr=0.0001, module__neurons=128; total time=   0.8s
[CV] END ......................lr=0.001, module__neurons=128; total time=   0.8s
[CV] END ......................lr=0.001, module__neurons=128; total time=   0.8s
[CV] END ............

ValueError: 
All the 27 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
27 fits failed with the following error:
Traceback (most recent call last):
  File "/home/miki/.local/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/miki/.local/lib/python3.14/site-packages/skorch/regressor.py", line 85, in fit
    return super(NeuralNetRegressor, self).fit(X, y, **fit_params)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/home/miki/.local/lib/python3.14/site-packages/skorch/net.py", line 1336, in fit
    self.partial_fit(X, y, **fit_params)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/home/miki/.local

In [ ]:
print(len(X_train))
print(len(y_train[:,0]))